# ASK Audio Demo on PYNQ-Z2

This notebook loads `ask_audio.bit`, initializes the onboard ADAU1761 codec, loads baseband symbols into IFM BRAM, and starts the ASK modulator. Probe the PYNQ-Z2 headphone output with an oscilloscope.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if (repo_root / 'pynq').exists():
    sys.path.insert(0, str(repo_root / 'pynq'))
else:
    sys.path.insert(0, str(Path.cwd()))

from ask_audio_demo import AskAudioDemo, pattern_symbols, print_addressable_ips, plot_debug_capture, save_debug_capture_csv, summarize_debug_capture
from pynq import Overlay

BITFILE = 'ask_audio.bit'


In [ ]:
# Diagnostic: verify the .hwh exposes both ask_audio_top_0 and axi_bram_ctrl_0.
ol_check = Overlay(BITFILE)
print_addressable_ips(ol_check)


In [ ]:
# Run this setup cell once after programming the overlay.
demo = AskAudioDemo(BITFILE, init_codec=True)
demo.stop()
{
    'ask_ip': demo.ask_ip_name,
    'ifm_bram': demo.ifm_bram_ip_name,
    'ifm_word_capacity': demo.ifm_bram_word_capacity,
    'codec_init': demo.codec_result,
}


In [ ]:
def run_ask_demo(carrier_hz, symbol_rate, symbol_count, pattern, *, loop=True, verify=True):
    symbols = pattern_symbols(symbol_count, pattern)
    demo.stop()
    load_verify = demo.load_symbols(symbols, verify=verify)
    config = demo.configure(
        carrier_hz=carrier_hz,
        symbol_rate=symbol_rate,
        symbol_count=len(symbols),
    )
    demo.start(loop=loop)
    return {
        'config': config,
        'pattern': [int(symbol) & 0x3 for symbol in pattern],
        'bram_load_verify': load_verify,
        'first_16_symbols': demo.read_symbols(min(16, len(symbols))),
        'status': demo.status(),
        'debug': demo.read_debug(),
    }


def capture_and_plot(count=1000, interval_s=0.005, csv_path='ask_debug_capture.csv'):
    debug_samples = demo.capture_debug_samples(count=count, interval_s=interval_s)
    if csv_path:
        save_debug_capture_csv(debug_samples, csv_path)
    print(summarize_debug_capture(debug_samples))
    fig, axes = plot_debug_capture(debug_samples)
    return debug_samples, fig, axes


In [ ]:
# Fast-rerun config cell. Change only this cell for most demos.
# Keep CARRIER_HZ low enough for software polling to draw a clean waveform.
CARRIER_HZ = 3
SYMBOL_RATE = 0.5
SYMBOL_COUNT = 32
PATTERN = [0, 1, 3, 2]

CAPTURE_COUNT = 2000
CAPTURE_INTERVAL_S = 0.005

run_result = run_ask_demo(CARRIER_HZ, SYMBOL_RATE, SYMBOL_COUNT, PATTERN)
run_result


In [ ]:
demo.status(), demo.read_debug()


In [ ]:
# Hardware register capture plot. Re-run after changing the config cell above.
debug_samples, fig, axes = capture_and_plot(
    count=CAPTURE_COUNT,
    interval_s=CAPTURE_INTERVAL_S,
)
fig


Oscilloscope starting point:

- Probe the headphone output, AC-coupled first.
- Time base: 1 ms/div to 5 ms/div.
- Expected carrier: 4 kHz.
- Expected envelope change: 100 symbols/s using the repeating 00, 01, 11, 10 pattern.
- If output is silent, check `demo.read_debug()` and use ILA on `ask_out`, `symbol_valid_dbg`, `codec_lrclk`, and `codec_sdata_o`.